In [ ]:
import pandas as pd
import numpy as np

In [ ]:
csv_path = '../data/clinical/clinical_data.csv'                                               # Ruta a los datos clínicos originales en formato CSV

In [ ]:
df = pd.read_csv(csv_path)                                                                    # Cargar los datos en un DataFrame de pandas    
df.head()                                                                                     # Mostrar las primeras filas del DataFrame

In [ ]:
df = df.dropna(how='all')                                                                     # Eliminar filas que contienen solo valores nulos
df = df.dropna(subset=["# IDENTIFICACION INTERNO DEL REGISTRO (ACTUAL)"])                     # Eliminar filas sin ID de registro
bi_rads = {
    'n.a.': np.nan,
    'n.a': np.nan,
}
df["Bi-RADS"] = df["Bi-RADS"].replace(bi_rads)                                                # Reemplazar valores no aplicables en Bi-RADS por NaN
df = df.dropna(subset=["Bi-RADS"])                                                            # Eliminar filas sin categoría Bi-RADS    


In [ ]:
columns_to_drop = [
    "FECHA DE NACIMIENTO",
    "Fecha de mamografía Reporte          DD-MM-AAAA",
    "Nodulos",
    "Morfologia de los nodulos",
    "Margenes Nodulos",
    "Densidad Nodulo",
    "Presencia Microcalcificaciones",
    "Calcificaciones tipicamente benignas",
    "Calcificaciones morfologia sospechosa",
    "Distribucion de las calcificaciones",
    "Presencia de asimetrias",
    "Tipo de asimetria",
    "Hallazgos asociados",
    "FECHA ESTUDIO                                                DD-MM-AAAA",
    "MODALIDAD BIOPSIA",
    "Lesion Mamaria benigna no proliferativa",
    "lesion Mamaria benigna proliferativa sin atipia",
    "Lesion Mamaria benigna proliferativa con atipia",
    "Lesion Mamaria Maligna Carcinoma in situ",
    "Lesion Mamaria Maligna Invasiva ",
    "ANOTACIONES",
    "OBSERVACIONES ",
    "Unnamed: 48",
    "Novedades del Etiquetado",
    "Lateralidad Maligno",
    "Lateralidad Benigno",
    "Raza",
    "Antecedente de CLIS (carcinoma lobullillar in situ)",
    "Antecedente de HDA o HLA (hiperplasia ductal atipica/hiperplasia lobulillar atipica)",
    "Antecedente radiacion Toráx antes de los 30 años"
]
df = df.drop(columns=columns_to_drop, axis=1)                                                     # Eliminar columnas innecesarias o con mucha información faltante

## Imputacion  y limpieza de datos

In [ ]:
df.rename(columns={"Indice de masa corporal (Kg.m2) (2 decimales)": "IMC"}, inplace=True)         # Renombrar columna para facilitar su manejo

indice_masa_corporal = {
    "s.d": np.nan,
    "d.s": np.nan
}                                                                                                 # Diccionario para reemplazar valores no numéricos por NaN                        
df['IMC'] = (
    df['IMC']
    .replace(indice_masa_corporal)
    .str.replace(',', '.', regex=False)
    .astype(float)
)                                                                                                 # Limpiar y convertir a float la columna de índice de masa corporal
media_indice_masa_corporal = df['IMC'].mean()                                                     # Calcular la media del índice de masa corporal
df['IMC'] = df['IMC'].fillna(media_indice_masa_corporal)                                          # Imputar valores faltantes con la media

In [ ]:
antecentes_tabaquismo = {
    "0- No": 0,
    "0. No": 0,
    "1- Si": 1,
    "1. Si": 1,
    "2. Desconocido": 0
}                                                                                                  # Diccionario para mapear antecedentes de tabaquismo a valores numéricos
df['Antecedentes de Tabaquismo'] = df['Antecedentes de Tabaquismo'].replace(antecentes_tabaquismo) # Mapear valores de antecedentes de tabaquismo

In [ ]:
antecedente_consumo_alcohol = {
    "0- No": 0,
    "0. No": 0,
    "1- Si": 1,
    "1. Si": 1,
    "2. Desconocido": 0,
    "2- Desconocido": 0
}                                                                                                                 # Diccionario para mapear antecedentes de consumo de alcohol a valores numéricos
df["Antecedente consumo de alcohol"] = df["Antecedente consumo de alcohol"].replace(antecedente_consumo_alcohol)  # Mapear valores de antecedentes de consumo de alcohol

In [ ]:
edad_menarquia = {
    "1- 7 a 11 años": "7 a 11 años",
    "1. 7 a 11 años": "7 a 11 años",
    "2- 12 a 13 años": "12 a 13 años",
    "2. 12 a 13 años": "12 a 13 años",
    "3- 14 o más años": "14 o más años",
    "3. 14 o más años": "14 o más años"
}                                                                                   # Diccionario para normalizar valores de edad de menarquia
df["Edad Menarquia"] = df["Edad Menarquia"].replace(edad_menarquia)                 # Normalizar valores de edad de menarquia
df['Edad Menarquia'] = df['Edad Menarquia'].fillna(df['Edad Menarquia'].mode()[0])  # Imputar valores faltantes con la moda

In [ ]:
menopausia_tardia = {
    "1. Si": "Si",
    "1- Si": "Si",
    "0. No": "No",
    "0- No": "No",
    "2. Desconocido": "Desconocido",
    "2- Desconocido": "Desconocido",
    "Desconocido": "Desconocido"
}                                                                                                                                                   # Diccionario para normalizar valores de menopausia tardia
df["menopausia tardia (mayor a 55 años)"] = df["menopausia tardia (mayor a 55 años)"].replace(menopausia_tardia)                                    # Normalizar valores de menopausia tardia
df['menopausia tardia (mayor a 55 años)'] = df['menopausia tardia (mayor a 55 años)'].fillna(df['menopausia tardia (mayor a 55 años)'].mode()[0])   # Imputar valores faltantes con la moda

In [ ]:
edad_1er_parto = {
    "1. No embarazos": "No embarazos",
    "1- No embarazos": "No embarazos",
    "2. < 20 años": "< 20 años",
    "2- < 20 años": "< 20 años",
    "3. 20-24 años": "20-24 años",
    "3- 20-24 años": "20-24 años",
    "4. 25-29 años": "25-29 años",
    "4- 25-29 años": "25-29 años",
    "5. > 30 años": "> 30 años",
    "5- > 30 años": "> 30 años",
    "6. Desconocido": "Desconocido",
    "6- Desconocido": "Desconocido"
}                                                                                       # Diccionario para normalizar valores de edad del primer parto
df["Edad 1er parto"] = df["Edad 1er parto"].replace(edad_1er_parto)                     # Normalizar valores de edad del primer parto
df['Edad 1er parto'] = df['Edad 1er parto'].fillna(df['Edad 1er parto'].mode()[0])      # Imputar valores faltantes con la moda

In [ ]:
antecedente_CDIS = {
    '1. Si': 1,
    '0. No': 0,
    '0- No': 0,
    '2- Desconocido': 0,
    '2. Desconocido': 0,
}                                                                                                                                       # Diccionario para mapear antecedentes de CDIS a valores numéricos
df["Antecedente de CDIS (carcinoma ductal in situ)"] = df["Antecedente de CDIS (carcinoma ductal in situ)"].replace(antecedente_CDIS)   # Mapear valores de antecedentes de CDIS
df['Antecedente de CDIS (carcinoma ductal in situ)'] = df['Antecedente de CDIS (carcinoma ductal in situ)'].fillna(df['Antecedente de CDIS (carcinoma ductal in situ)'].mode()[0]) #  Imputar valores faltantes con la moda


In [ ]:
df.rename(columns={"LUGAR DE RESIDENCIA": "Residencia"}, inplace=True)                               # Renombrar columna de lugar de residencia
residencia = {
    '1. Urbana': "Urbana",
    '1- Urbana': "Urbana",
    '2. Rural': "Rural",
    '2- Rural': "Rural",
}                                                                                                    # Diccionario para mapear lugar de residencia
df["Residencia"] = df["Residencia"].replace(residencia)                                              # Mapear valores de lugar de residencia

In [ ]:
afiliacion = {
    '1. Subsidiado' : "Subsidiado",
    '1- Subsidiado' : "Subsidiado",
    '2. Contributivo' : "Contributivo",
    '2- Contributivo' : "Contributivo",
    '3. Especial' : "Especial",
}                                                                                                    # Diccionario para mapear tipo de afiliación
df["Tipo  de afiliacion"] = df["Tipo  de afiliacion"].replace(afiliacion)                            # Mapear valores de tipo de afiliación
df.rename(columns={"Tipo  de afiliacion": "Tipo de afiliacion"}, inplace=True)                       # Renombrar columna de tipo de afiliación

In [ ]:
escolaridad = {
    '1. Primaria': "Primaria",
    '1- Primaria': "Primaria",
    '2. Bachillerato': "Bachillerato",
    '2- Bachillerato': "Bachillerato",
    '3. Tecnico': "Tecnico",
    '3- Tecnico': "Tecnico",
    '4. Universitario': "Universitario",
    '4- Universitario': "Universitario",
    '5. Posgrado': "Posgrado",
    '5- Posgrado': "Posgrado",
    '6- Ninguno': "Ninguno",
    '6. Ninguno': "Ninguno",
    '7. Desconocido': "Desconocido",
    '7- Desconocido': "Desconocido",
}                                                                                                     # Diccionario para mapear niveles de escolaridad
df["Escolaridad"] = df["Escolaridad"].replace(escolaridad)                                            # Mapear valores de escolaridad

In [ ]:
antecedente_CDI = {
    '1. Si': 1,
    '1- Si': 1,
    '0. No': 0,
}
df["Antecedente de CDI (Carcinoma Ductal Infiltrante)"] = df["Antecedente de CDI (Carcinoma Ductal Infiltrante)"].replace(antecedente_CDI)   # Mapear valores de antecedentes de CDI
df['Antecedente de CDI (Carcinoma Ductal Infiltrante)'] = df['Antecedente de CDI (Carcinoma Ductal Infiltrante)'].fillna(df['Antecedente de CDI (Carcinoma Ductal Infiltrante)'].mode()[0])     # Imputar valores faltantes con la moda

In [ ]:
cli = {
    '1. Si': 1,
    '0- No': 0,
    '0. No': 0,
}
df["CLI (Carcinoma Lobulillar Infiltrante)"] = df["CLI (Carcinoma Lobulillar Infiltrante)"].replace(cli)   # Mapear valores de antecedentes de CLI
df['CLI (Carcinoma Lobulillar Infiltrante)'] = df['CLI (Carcinoma Lobulillar Infiltrante)'].fillna(df['CLI (Carcinoma Lobulillar Infiltrante)'].mode()[0]) 

In [ ]:
antecedente_TRH = {
    '1. Si': 1,
    '1- Si': 1,
    '0. No': 0,
    '0- No': 0,
}
df["Antecedente de TRH"] = df["Antecedente de TRH"].replace(antecedente_TRH)                        # Mapear valores de antecedentes de TRH
df['Antecedente de TRH'] = df['Antecedente de TRH'].fillna(df['Antecedente de TRH'].mode()[0])      # Imputar valores faltantes con la moda

In [ ]:
antecedente_mutacion = {
    '1. Si': "Si",
    '0. No': "No",
    '0- No': "No",
    '2. Desconocido': "Desconocido",
    '2- Desconocido': "Desconocido",
}
df["Antecedente de mutacion BRCA 1 ó 2, Sindrome genetico con riesgo para Ca de mama-"] = df["Antecedente de mutacion BRCA 1 ó 2, Sindrome genetico con riesgo para Ca de mama-"].replace(antecedente_mutacion)   # Mapear valores de antecedentes de mutacion BRCA 1 ó 2

In [ ]:
antecedente_Bx = {
    '1. Si': 1,
    '0. No': 0,
    '0- No': 0,
}
df["Antecedente de Bx mama benigna"] = df["Antecedente de Bx mama benigna"].replace(antecedente_Bx)   # Mapear valores de antecedentes de biopsia mamaria benigna

In [ ]:
Bx_realizadas = {
    '1. 1': "1",
    '2. 2 ó más': "2 ó más",
    '3. Desconocido ': "Desconocido",
    '2- Desconocido': "Desconocido",
}
df["# Bx realizadas"] = df["# Bx realizadas"].replace(Bx_realizadas)                                  # Mapear valores de número de biopsias realizadas
df['# Bx realizadas'] = df['# Bx realizadas'].fillna(df['# Bx realizadas'].mode()[0])                 # Imputar valores faltantes con la moda

In [ ]:
familiares = {
    '1. Ninguna ': "Ninguna",
    '1- Ninguna': "Ninguna",
    '2. Una': "1",
    '3. Más de una': "Más de 1",
    '3. Más de una ': "Más de 1",
}
df["# Familiares en 1er grado con Ca de mama-"] = df["# Familiares en 1er grado con Ca de mama-"].replace(familiares)   # Mapear valores de número de familiares en primer grado con cáncer de mama
df.rename(columns={"# Familiares en 1er grado con Ca de mama-": "Familiares en 1er grado con Ca de mama-"}, inplace=True)

In [ ]:
lateralidad_hallazgo = {
    '1- DERECHO': "Derecho",
    '1. DERECHO': "Derecho",
    '1- IZQUIERDO': "Izquierdo",
    '2- IZQUIERDO': "Izquierdo",
    '2. IZQUIERDO': "Izquierdo",
    'n.a': "No se especifica",
    'n.a.': "No se especifica",
    '3. BILATERAL': "Bilateral"
}
df["LATERALIDAD HALLAZGO"] = df["LATERALIDAD HALLAZGO"].replace(lateralidad_hallazgo)                    # Mapear valores de lateralidad del hallazgo
df.rename(columns={"LATERALIDAD HALLAZGO": "Lateralidad del hallazgo"}, inplace=True)                    # Renombrar columna de lateralidad del hallazgo

In [ ]:
df.rename(columns={"EDAD              (toma primera mamografía)": "Edad primera mamografía"}, inplace=True)        # Renombrar columna de edad en la primera mamografía
df.rename(columns={"# IDENTIFICACION INTERNO DEL REGISTRO (ACTUAL)": "ID"}, inplace=True)                          # Renombrar columna de ID de registro

In [ ]:
df.rename(columns={"Estrato socieconomico (1 a 6)": "Estrato socieconomico"}, inplace=True)             # Renombrar columna de estrato socioeconómico
estrato = {
    '1': 1,
    '2': 2,
    'Desconocido': np.nan,
    '3': 3,
    '4': 4,
}
df["Estrato socieconomico"] = df["Estrato socieconomico"].replace(estrato)                              # Mapear valores de estrato socioeconómico
df['Estrato socieconomico'] = df['Estrato socieconomico'].fillna(df['Estrato socieconomico'].mode()[0]) # Imputar valores faltantes con la moda


In [ ]:
def assign_diagnosis(birads):
    if birads in ["0", "1", "2", "3"]:
        return 0
    return 1

df['Diagnostico'] = df['Bi-RADS'].apply(assign_diagnosis)   # Asignar diagnóstico basado en Bi-RADS
df.drop(columns=["Bi-RADS"], axis=1, inplace=True)          # Eliminar columna Bi-RADS tras asignar diagnóstico

In [ ]:
df.rename(columns={"Antecedente de mutacion BRCA 1 ó 2, Sindrome genetico con riesgo para Ca de mama-": "Antecedente de mutacion BRCA"}, inplace=True)

In [ ]:
categorical_columns = [
                       'Estrato socieconomico',
                       'Residencia', 
                       'Tipo de afiliacion',
                       'Escolaridad', 
                       'Edad Menarquia',
                       'menopausia tardia (mayor a 55 años)', 
                       'Edad 1er parto',
                       'Antecedente de mutacion BRCA',
                       '# Bx realizadas',
                       'Familiares en 1er grado con Ca de mama-',
                       'Lateralidad del hallazgo'
                    ]


In [ ]:
df_depured = df.copy()                                                  # Crear una copia del DataFrame depurado antes de la codificación one-hot
df_encoded = pd.get_dummies(df_depured, columns=categorical_columns)    # Codificación one-hot de variables categóricas
bool_cols = df_encoded.select_dtypes('bool').columns                    # Seleccionar columnas de tipo booleano 
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)               # Convertir columnas booleanas a enteros

In [ ]:
output_csv_path = '../data/clinical/clinical_data_processed.csv'    # Guardar el DataFrame procesado en un nuevo archivo CSV
df_encoded.to_csv(output_csv_path, index=False)                     # Guardar el DataFrame procesado en un nuevo archivo CSV